**Before starting this practical session**:

1. **Change the runtime type on Google Colab**: `Runtime > Change runtime type > T4 GPU`
2. **Install the packages below**:

In [ ]:
! pip install lightning torchmetrics torchinfo

3. Run this code to suppress some messages and warnings that may be displayed.

In [ ]:
import logging
logging.getLogger("lightning").setLevel(logging.ERROR)
logging.getLogger("lightning.pytorch.utilities.rank_zero").setLevel(logging.WARNING)
logging.getLogger("lightning.pytorch.accelerators.cuda").setLevel(logging.WARNING)
logger = logging.getLogger("lightning")
logger.propagate = False

import warnings
warnings.filterwarnings("ignore", ".*does not have many workers.*")
warnings.filterwarnings("ignore", ".*Missing logger folder.*")
warnings.filterwarnings("ignore", "`isinstance\\(treespec, LeafSpec\\)` is deprecated.*")
warnings.filterwarnings("ignore", ".*You are using `torch.load` with `weights_only=False`.*")

# Forecasting electricity consumption

We will work on a dataset called [short-term electricity load forecasting](https://data.mendeley.com/datasets/byx7sztj59/1).
The goal is to predict electricity consumption in Panama at a given moment.
The available data spans from January 3, 2015 to June 26, 2020 at an hourly scale (one observation every hour) and includes not only electricity consumption but also weather data for geographic areas of Panama and indicators for public holidays and school days.

## (Down)loading the dataset

The `load_dataset` function loads (and downloads if needed) the dataset and returns a [`pandas.DataFrame()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html) object with the data.

In [ ]:
import pandas as pd


def load_dataset(path="data"):
    """Load the dataset.

    Parameters
    ----------
    path : str
        Path to the directory. If not found, the file will be downloaded and
        put in this directory.

    Returns
    -------
    df : DataFrame
        Data.

    """
    import os
    from urllib.request import urlretrieve

    if not os.path.exists(path):
        os.makedirs(path)

    file = "continuous-dataset.csv"

    # Download the file if necessary
    if not os.path.isfile(os.path.join(path, file)):
        url = (
            "https://raw.githubusercontent.com/johannfaouzi/smart-data-deep-learning/"
            "main/data/electricity-load/continuous-dataset.csv"
        )
        urlretrieve(url, os.path.join(path, file))

    return pd.read_csv(os.path.join(path, file), index_col=0, parse_dates=[0])

Simply run the function to (down)load the dataset:

In [ ]:
df = load_dataset()

Let's visualize the dataset:

In [ ]:
df

There are $48\ 048$ observations and $16$ variables:
* `nat_demand` corresponds to national electricity consumption.
* `T2M_toc`, `T2M_san`, and `T2M_dav` correspond to the temperature at two meters in Tocumen, Santiago, and David respectively.
* `QV2M_toc`, `QV2M_san`, and `QV2M_dav` correspond to the humidity at two meters in Tocumen, Santiago, and David respectively.
* `TQL_toc`, `TQL_san`, and `TQL_dav` correspond to the precipitation level in Tocumen, Santiago, and David respectively.
* `W2M_toc`, `W2M_san`, and `W2M_dav` correspond to the wind speed at two meters in Tocumen, Santiago, and David respectively.
* `Holiday_ID` is an integer indicating the type of public holiday (23 unique values)
* `holiday` is a binary variable indicating whether the day is a public holiday or not.
* `school` is a binary variable indicating whether the day is a school day or not.

The main goal of this notebook is to illustrate the use of recurrent neural networks and not to obtain the best possible model.
We will therefore preprocess the data to make it easier to use recurrent neural networks.

First, we will remove the `Holiday_ID` column, not because it is irrelevant, but simply so that we don't have to process it.
To simplify the task and remove the daily seasonality, we will also resample the dataset and take the average value of the variables over each day.

In [ ]:
df = df.drop("Holiday_ID", axis=1).resample("D").mean()
df

Only $2003$ observations and $15$ variables remain.

Let's visualize the time series whose future values we want to predict, i.e., `nat_demand`:

In [ ]:
df["nat_demand"].plot();

We observe a drop in electricity consumption shortly after the beginning of 2020, most likely related to the Covid-19 pandemic.
To avoid evaluating the model on this period, we will limit ourselves to data up to the end of 2019.

In [ ]:
df = df.loc[df.index.year <= 2019]

In [ ]:
df.shape

The final dataset therefore contains $1824$ observations and $15$ variables.

## Training, validation, and test sets - Data normalization

Splitting the full dataset into training, validation, and test sets is naturally different for temporal data since observations are not independent of one another.
The split is also done at the temporal level.
Schematically, training corresponds to the past, validation to the present, and test to the future.

The `DataFrame` is already chronologically ordered.
It is therefore necessary to define two cutoff dates to separate the training set from the validation set on one hand, and the validation set from the test set on the other hand.
We will use the years 2015 to 2017 for the training set, the year 2018 for the validation set, and the year 2019 for the test set.

You have probably noticed that the different variables in the `DataFrame` do not have the same scales: the temperature variables, for example, have much higher values than those for humidity and precipitation.
Such scale differences can complicate the training of a machine learning model, and this is even more true for an artificial neural network.
We will therefore normalize the data.
However, the validation and test sets must not be used for training, and estimating the parameters used for normalization is part of training.
It is therefore necessary to use only the training data to estimate these parameters.
The binary variables do not need to be normalized.

It is time to write the code to perform this work.
Since the data is already chronologically ordered, we simply need to find the indices of the first day of the years 2018 and 2019 in order to perform the split into three sets.

In [ ]:
import numpy as np


# Index corresponding to January 1, 2018
first_day_2018_idx = np.where(df.index.year == 2018)[0].min()

# Index corresponding to January 1, 2019
first_day_2019_idx = np.where(df.index.year == 2019)[0].min()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import torch


# Get the data as a NumPy array
X_numpy = df.to_numpy()

# Keep the variables to scale
X_numpy_to_scale = X_numpy[:, :-2]

# Scale these variables
scaler = MinMaxScaler(feature_range=(-1.0, 1.0))
scaler.fit(X_numpy_to_scale[:first_day_2018_idx])
X_numpy_scaled = scaler.transform(X_numpy_to_scale)

# Parameters of the normalization of the time series of interest (first column),
# needed to denormalize the predictions of the model
scaler_params = {
    "data_min": float(scaler.data_min_[0]),
    "data_max": float(scaler.data_max_[0]),
    "feature_range": tuple(scaler.feature_range),
}

# Concatenate the scaled variables with the binary variables
X_numpy_full = np.c_[X_numpy_scaled, X_numpy[:, -2:]]

# Transform this NumPy array into a PyTorch tensor
X = torch.from_numpy(X_numpy_full).to(dtype=torch.float32)

# Delete the temporary variables
del X_numpy, X_numpy_to_scale, X_numpy_scaled, X_numpy_full

## `Dataset` and `Dataloader` objects

Since we have here a dataset in a particular context (time series), we will need to create our own version of `Dataset`, i.e., create our own class inheriting from the [`torch.utils.data.Dataset()`](https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset) class.

In the dataset documentation, it is recommended to have at least $72$ hours between the input data and the output data.
Indeed, some minimum amount of advance notice is needed to change the future quantity of electricity to produce.
Since we resampled the hourly data into daily data, the minimum gap between the last day of the input and the first day of the output is therefore $3$ days.
The `time_gap` variable defines the gap (in number of days) between the last day of the input and the first day of the output, and its default value is $3$.

We also need to define the length of the input and output time series.
The `output_length` variable determines the number of days for which we predict electricity consumption.
Regarding the outputs, we also do not seek to predict electricity consumption too far in advance.
We will therefore predict it for the next $7$ days (after the delay defined by `time_gap`).
The `input_length` variable determines the number of days for which we provide the input data.
The number of days could be variable: we are allowed to use all past data to predict the future.
However, to reduce the complexity of training and facilitate the creation of batches of observations, we will use a fixed number of days, set to $30$, i.e., roughly a month.
An obvious limitation of this approach is that we do not model annual seasonality.

The `create_train_val_test_splits()` function defined below creates the training, validation, and test sets based on the values provided for the different arguments.

In [ ]:
from torch.utils.data import Dataset, DataLoader


def create_train_val_test_splits(
    X, train_val_split_idx, val_test_split_idx, input_length=30, output_length=7, time_gap=3
):
    """Create the training, validation, and test sets.

    Parameters
    ----------
    train_val_split_idx : int
        Index splitting the training and validation sets.

    val_test_split_idx : int
        Index splitting the validation and test sets.

    input_length : int (default = 30)
        Number of days for the input time series.

    output_length : int (default = 7)
        Number of days to forecast.

    time_gap : int (default = 3)
        Number of days between the last day of the input and the first day of the predicted output.

    Returns
    -------
    dataset_train : Dataset
        Training set.

    dataset_val : Dataset
        Validation set.

    dataset_test : Dataset
        Test set.
    """

    class CustomDataset(Dataset):
        """
        Parameters
        ----------
        X : Tensor, shape = (n_samples, n_features)
            Data.

        dataset : {"training", "validation", "test"}
            Dataset.
        """
        def __init__(self, X, dataset):
            if dataset not in ("training", "validation", "test"):
                raise ValueError("'dataset' must be one of 'training', 'validation' or 'test'.")
            self.X = X
            self.dataset = dataset

        def __len__(self):
            if self.dataset == "training":
                return train_val_split_idx - (input_length + time_gap + output_length) + 1
            elif self.dataset == "validation":
                return (val_test_split_idx - (train_val_split_idx + output_length)) // output_length + 1
            else:
                return (len(X) - (val_test_split_idx + output_length)) // output_length + 1

        def __getitem__(self, idx):
            if not (0 <= idx < len(self)):
                raise ValueError(
                    f"Invalid index. The size of the dataset is {len(self)}."
                )
            if self.dataset == "training":
                start_idx = 0
                step = 1
            elif self.dataset == "validation":
                start_idx = train_val_split_idx - input_length - time_gap
                step = output_length
            else:
                start_idx = val_test_split_idx - input_length - time_gap
                step = output_length

            start_idx_input = start_idx + idx * step
            end_idx_input = start_idx_input + input_length

            start_idx_output = end_idx_input + time_gap
            end_idx_output = start_idx_output + output_length

            return self.X[start_idx_input:end_idx_input], self.X[start_idx_output:end_idx_output, 0]

    return CustomDataset(X, "training"), CustomDataset(X, "validation"), CustomDataset(X, "test")

We run it to create the three datasets, then we create the *dataloaders* for each of the datasets.

In [ ]:
input_length = 30
output_length = 7
time_gap = 3

dataset_train, dataset_val, dataset_test = create_train_val_test_splits(
    X, first_day_2018_idx, first_day_2019_idx, input_length, output_length, time_gap
)

dataloader_train = DataLoader(dataset_train, batch_size=32, shuffle=True)
dataloader_val = DataLoader(dataset_val, batch_size=52, shuffle=False)
dataloader_test = DataLoader(dataset_test, batch_size=52, shuffle=False)

## Recurrent neural network

It is finally time to create our recurrent neural network! The architecture of your network will be the following sequential architecture:
* LSTM layer with `lstm_hidden_size` variables in the hidden state. We only keep the last hidden state.
* Linear layer with `linear_out_features` output variables.
* ReLU activation function.
* Linear layer with `output_length` output variables (since we predict for the next `output_length` days with a gap of `time_gap` days).

The point of using parameters for these variables is to be able to try different combinations without needing to recreate a new class each time.
However, we will give default values to the parameters: `lstm_hidden_size=256` and `linear_out_features=128`.
The constructor also takes as arguments the number of days to forecast (`output_length`, $7$ by default) and the parameters of the normalization of the time series of interest (`data_min`, `data_max`, and `feature_range`), which are used to denormalize the predictions (more details below). These parameters are stored in the `scaler_params` dictionary defined above, so you can simply create a model with `RecurrentNeuralNetwork(**scaler_params)`.

### Question 1

Complete the `__init__()` and `forward()` methods of the `RecurrentNeuralNetwork` class defined below. Refer to the documentation of [`torch.nn.LSTM()`](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html) to understand the expected input and the output returned by this layer.

As a reminder, the input returned by the *dataloader*, i.e., the input of the model (the `x` argument of the `forward()` method), is a tensor of size $(N, L, H_{in})$ where:
* $N$ is the batch size (the `batch_size` argument of the dataloader).
* $L$ is the length of each sequence (i.e., the `input_length` variable defined above).
* $H_{in}$ is the dimension at each time step, i.e., the number of variables, which is equal to $15$.

In [ ]:
import lightning as L
from torchmetrics import MeanSquaredError


class RecurrentNeuralNetwork(L.LightningModule):  # Always inherit from the lightning.LightningModule class
    def __init__(
        self, data_min, data_max, feature_range=(-1.0, 1.0),
        lstm_hidden_size=256, linear_out_features=128, output_length=7
    ):
        """Constructor.

        In the constructor, we always run the constructor of the parent class and we
        instantiate all the elements of our network: the layers, the activation functions,
        the loss function and the metrics.

        Parameters
        ----------
        data_min, data_max : float
            Minimum and maximum values of the time series of interest on the training set
            (used to denormalize the predictions).

        feature_range : tuple[float, float] (default = (-1.0, 1.0))
            Range of the normalized values.

        lstm_hidden_size : int (default = 256)
            Number of variables in the hidden state of the LSTM layer.

        linear_out_features : int (default = 128)
            Number of output variables of the first linear layer.

        output_length : int (default = 7)
            Number of days to forecast.
        """
        super().__init__()  # Always run the constructor of the parent class

        # Save the hyperparameters (required to reload the model from a checkpoint)
        self.save_hyperparameters()

        # Parameters of the normalization of the time series of interest
        self.data_min = data_min
        self.data_max = data_max
        self.feature_range = feature_range

        ### BEGIN TODO ###

        #### END TODO ####

        self.loss = MeanSquaredError()

    def forward(self, x):
        """Implements the forward pass."""
        ### BEGIN TODO ###
        # y =
        ### END TODO ###
        return y

    def step(self, batch):
        """Performs a step.

        A step consists in mapping a batch of samples to the loss function for this batch of input samples.

        Parameters
        ----------
        batch : tuple
            A batch of samples.

        dataset : {"training", "validation", "test"}
            Data set used.

        Returns
        -------
        loss : Tensor, shape = (1,)
            Loss function evaluated for this batch of samples.
        """
        x, y = batch  # x corresponds to the inputs, y corresponds to the outputs
        y_pred = self(x)  # Forward pass outputting the forecasted values
        loss = self.loss(y_pred, y)  # Evaluation of the loss function

        # Predictions in the original space
        y_cpu = y.cpu().detach()
        y_pred_cpu = y_pred.cpu().detach()
        y_rescaled = self.inverse_transform(y_cpu)
        y_pred_rescaled = self.inverse_transform(y_pred_cpu)
        mse_rescaled = self.loss(y_pred_rescaled, y_rescaled)

        return loss, mse_rescaled

    def training_step(self, batch):
        """Performs a training step."""
        loss, mse_rescaled = self.step(batch)
        self.log("loss_train", loss, on_step=False, on_epoch=True)
        self.log("mse_train", mse_rescaled, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch):
        """Performs a validation step."""
        loss, mse_rescaled = self.step(batch)
        self.log("loss_val", loss, on_step=False, on_epoch=True)
        self.log("mse_val", mse_rescaled, on_step=False, on_epoch=True)
        return loss

    def test_step(self, batch):
        """Performs a test step."""
        loss, mse_rescaled = self.step(batch)
        self.log("loss_test", loss, on_step=False, on_epoch=True)
        self.log("mse_test", mse_rescaled, on_step=False, on_epoch=True)
        return loss

    def on_train_start(self):
        """Code to run at the beginning of training."""
        string = f"Version {self.trainer.logger.version}"
        print(f"{string}\n{'=' * len(string)}\n")

    def on_train_epoch_end(self):
        """Code to run at the end of each training epoch."""
        metrics = self.trainer.callback_metrics
        string = (f"""
            Epoch {self.trainer.current_epoch + 1} / {self.trainer.max_epochs}
            ------------------------------------------
            |  Data set  | Loss function |    MSE    |
            | ---------- | ------------- | --------- |
            |  Training  |{metrics['loss_train'].item():^15.5f}|{metrics['mse_train'].item():^11.2f}|
            | Validation |{metrics['loss_val'].item():^15.5f}|{metrics['mse_val'].item():^11.2f}|
            ------------------------------------------
        """)
        string = "\n".join([line.strip() for line in string.strip().split("\n")])
        print(string, "\n")

    def configure_optimizers(self):
        """Configure the optimization algorithms."""
        optimizer = torch.optim.Adam(self.parameters())
        return optimizer

    def inverse_transform(self, x):
        """Applies the inverse transform for the time series of interest.

        Parameters
        ----------
        x : Tensor
            Scaled time series.

        Returns
        -------
        x_new : Tensor
            Unscaled time series.
        """
        return (
            (x - self.feature_range[0])
            /
            (self.feature_range[1] - self.feature_range[0])
            *
            (self.data_max - self.data_min)
        ) + self.data_min

### Question 2

Display a summary of your neural network with the default values. How many trainable parameters does it have?

In [ ]:
# TODO

**Answer**: TODO

We will now train the model for `100` epochs by running the code below.

In [ ]:
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import TQDMProgressBar


model = RecurrentNeuralNetwork(**scaler_params)

trainer = L.Trainer(
    max_epochs=100,
    enable_model_summary=False,  # disable model summary
    logger=CSVLogger('.'),  # save the results in a CSV file
    num_sanity_val_steps=0,  # do not perform any validation step before training
    callbacks=[TQDMProgressBar(refresh_rate=0)],  # remove the progress bar
    log_every_n_steps=0,  # save the results at the end of each epoch only
)

trainer.fit(
    model=model,
    train_dataloaders=dataloader_train,
    val_dataloaders=dataloader_val
)

### Question 3

As a reminder, the mean squared error is both the loss function (for training the model) and the metric (for evaluating the model).
However, we normalized the data to make training the model easier.
For the metric, we will *denormalize* the predictions to put them back in the original space, since that is what matters in practice.
This is the purpose of the `inverse_transform()` method of the `RecurrentNeuralNetwork()` class.

Visualize the model's performance on the training and validation sets using the `plot_loss_mse()` function defined below.
Do you notice any issue?

In [ ]:
import matplotlib.pyplot as plt


def plot_loss_mse(savedir=".", version=None):
    """Plot the loss and MSE curves.

    Parameters
    ----------
    savedir : str (default = ".")
        Path to the directory with the saved results.

    version : int or None (default = None)
        Version number of the model.
    """
    # Get the results as a pandas.DataFrame
    import os
    if version is None:
        version = max([
            int(folder.split("version_")[1])
            for folder in os.listdir(os.path.join(savedir, "lightning_logs"))
            if folder.startswith("version")
        ])
    df = pd.read_csv(os.path.join(savedir, "lightning_logs", f"version_{version}", "metrics.csv"))
    df["epoch"] += 1  # Start counting the epochs from 1 instead of 0

    loss_train = df.dropna(subset="loss_train").set_index("epoch")["loss_train"]
    loss_val = df.dropna(subset="loss_val").set_index("epoch")["loss_val"]

    mse_train = df.dropna(subset="mse_train").set_index("epoch")["mse_train"]
    mse_val = df.dropna(subset="mse_val").set_index("epoch")["mse_val"]

    # Plot the results
    plt.figure(figsize=(13, 4))

    plt.subplot(1, 2, 1)
    plt.plot(loss_train.index, loss_train.to_numpy(), "-", label="Training");
    plt.plot(loss_val.index, loss_val.to_numpy(), "-", label="Validation");
    plt.xlabel("Epoch")
    plt.ylabel("Loss function")
    plt.yscale("log")
    plt.legend();

    plt.subplot(1, 2, 2)
    plt.plot(mse_train.index, mse_train.to_numpy(), "-", label="Training");
    plt.plot(mse_val.index, mse_val.to_numpy(), "-", label="Validation");
    plt.xlabel("Epoch")
    plt.ylabel("Mean squared error")
    plt.yscale("log")
    plt.legend();

In [ ]:
plot_loss_mse()

**Answer**: TODO

### Question 4

Display the model's predictions and the mean squared error on the training and validation sets using the `plot_true_pred()` function defined below:

In [ ]:
def plot_true_pred(model, dataset):
    """Plot the true and forecasted time series on a dataset.

    Parameters
    ----------
    model : RecurrentNeuralNetwork
        Trained model.

    dataset : Dataset
        Dataset
    """
    if dataset.dataset == "training":
        step = 7
    else:
        step = 1

    # Obtain the forecasted values
    x_ = torch.stack([dataset[i][0] for i in range(0, len(dataset), step)])
    model.eval()
    with torch.no_grad():
        y_pred = model.inverse_transform(
            model(x_.to(model.device)).view(-1).cpu()
        ).numpy()

    # Obtain the true values
    y_true = model.inverse_transform(
        torch.cat([dataset[i][1] for i in range(0, len(dataset), step)])
    ).numpy()

    if dataset.dataset == "training":
        index = df.iloc[input_length+time_gap:first_day_2018_idx].index[:len(y_true)]
    elif dataset.dataset == "validation":
        index = df.iloc[first_day_2018_idx:first_day_2019_idx].index[:len(y_true)]
    else:
        index = df.iloc[first_day_2019_idx:].index[:len(y_true)]

    df_temp = pd.DataFrame([index, y_true, y_pred]).T.infer_objects()
    df_temp.columns = ["Date", "True values", "Predicted values"]
    df_temp.set_index("Date", inplace=True)

    plt.figure(figsize=(20, 4))
    plt.plot(df_temp["True values"], label="True values")
    plt.plot(df_temp["Predicted values"], label="Predicted values")
    plt.legend()
    plt.title(f"Mean squared error = {((y_true - y_pred) ** 2).mean():.2f}")

In [ ]:
# TODO

Now that we have noticed that the model suffers from overfitting, it is time to propose a method to avoid it.
One possible approach is to perform *early stopping*, i.e., stop the model's training earlier.
A classic stopping criterion is that the loss function on the validation set no longer decreases after $n$ epochs, where $n$ is a hyperparameter to define.

With PyTorch Lightning, this approach is defined in the `Trainer`, and more specifically through the `callbacks` argument.
We tell PyTorch Lightning that we want to perform early stopping by adding an instance of [`lightning.pytorch.callbacks.early_stopping.EarlyStopping()`](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.callbacks.EarlyStopping.html). In this instance, we indicate the score to monitor (`monitor` argument), which mode to use (`mode` argument, `"min"` if the score decreases when the model is better, `"max"` if the score increases when the model is better), and the number of checks (i.e., the number of epochs if a check is performed only at the end of each epoch) without improvement (`patience` argument).
Patience is important because it is possible to have a few epochs where the model no longer progresses, and then have it start progressing again afterward.

Here, the score to monitor is the loss function, which decreases as the model gets better.
We use the default value for patience, which is 3.
Run the code below:

In [ ]:
from lightning.pytorch.callbacks.early_stopping import EarlyStopping


model_early_stopping = RecurrentNeuralNetwork(**scaler_params)

trainer_early_stopping = L.Trainer(
    max_epochs=100,
    enable_model_summary=False,  # disable model summary
    logger=CSVLogger('.'),  # save the results in a CSV file
    num_sanity_val_steps=0,  # do not perform any validation step before training
    callbacks=[
        TQDMProgressBar(refresh_rate=0),  # remove the progress bar
        EarlyStopping(monitor="loss_val", mode="min", patience=3)  # perform early stopping
    ],
    log_every_n_steps=0,  # save the results at the end of each epoch only
)

trainer_early_stopping.fit(
    model=model_early_stopping,
    train_dataloaders=dataloader_train,
    val_dataloaders=dataloader_val
)

Notice that training stops well before reaching the maximum number of epochs ($100$).

### Question 5

Display the performance of this model on the training and validation sets using the `plot_true_pred()` function.
Compare it to that of the previous model.

In [ ]:
# TODO

We also notice that the final model, when *early stopping* occurred, corresponds to the one from the last epoch, which achieves a lower performance than the model from another, earlier epoch by definition.
By default, Lightning saves the model at the end of each epoch in the `version_X/checkpoints` directory (only the checkpoint of the last epoch is kept).
It is possible to change this behavior by again using the `callbacks` argument of the `Trainer` instance, using the [`lightning.pytorch.callbacks.ModelCheckpoint()`](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.callbacks.ModelCheckpoint.html) tool, providing it with which score to monitor.
We use the same score as for *early stopping*, i.e., the loss function on the validation set.

Run the code below:

In [ ]:
from lightning.pytorch.callbacks import ModelCheckpoint


model_early_stopping_best_checkpoint = RecurrentNeuralNetwork(**scaler_params)

trainer_early_stopping_best_checkpoint = L.Trainer(
    max_epochs=100,
    enable_model_summary=False,  # disable model summary
    logger=CSVLogger('.'),  # save the results in a CSV file
    num_sanity_val_steps=0,  # do not perform any validation step before training
    callbacks=[
        TQDMProgressBar(refresh_rate=0),  # remove the progress bar
        EarlyStopping(monitor="loss_val", mode="min", patience=3),  # perform early stopping
        ModelCheckpoint(monitor="loss_val")  # save the model with the lowest "loss_val" value
    ],
    log_every_n_steps=0,  # save the results at the end of each epoch only
)

trainer_early_stopping_best_checkpoint.fit(
    model=model_early_stopping_best_checkpoint,
    train_dataloaders=dataloader_train,
    val_dataloaders=dataloader_val
)

### Question 6

When training is finished, go to the `lightning_logs/version_X/checkpoints` folder (replace `X` with the version corresponding to this model, normally the latest one if it is the latest model trained) and look at the file name.
It indicates the epoch and the step (the step is the number of iterations, i.e., the number of epochs multiplied by the number of batches in the training set) of the saved model.
Verify that the epoch number in the file indeed corresponds to the epoch number with the lowest loss function (or MSE, which is the same here) on the validation set.

> **Warning**: Lightning counts epochs starting from $0$, but the displayed output counts epochs starting from $1$ (since it is more natural for a human to count starting from 1). It is therefore normal for the epoch number in the file to be 1 less than the best epoch number displayed in the output.

The model's trainable parameters are still those from the last epoch.
We load the saved trainable parameters, which give better performance on the validation set.

In [ ]:
import os


# Get the number of the latest version
version = max([
    int(folder.split("version_")[1])
    for folder in os.listdir(os.path.join(".", "lightning_logs"))
    if folder.startswith("version")
])
# Otherwise, you can provide the version number
# version =

# Get the path to the file
path = f"./lightning_logs/version_{version}/checkpoints"
file = os.listdir(path)[0]

# Create a model with the saved trainable parameters
model_early_stopping_best_checkpoint = RecurrentNeuralNetwork.load_from_checkpoint(
    os.path.join(path, file), weights_only=True, map_location=torch.device("cpu")
)

### Question 7

Choose the best model on the validation set and evaluate it on the test set.
Out of curiosity, also evaluate the other models.
Is the best model on the validation set also the best model on the test set?

In [ ]:
# TODO